# 01i — Extract WorldPop Population Density
**Data source:** [WorldPop Global 100m Population Density](https://www.worldpop.org/)

**Input:** `train_base.parquet`, `val_base.parquet` from notebook 00 output

**Output:** `worldpop.parquet` (one row per unique station)

**Estimated time:** ~5-10 min (downloading and raster masking)

> Enable Internet in Kaggle settings.

In [ ]:
# Install required packages if running in Kaggle environment
!pip install -q geopandas pyarrow requests tqdm rasterio shapely

In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
import rasterio
from rasterio.mask import mask
from shapely.geometry import Point
import matplotlib.pyplot as plt
import os, time, requests, logging
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# === Logging Setup ===
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)-5s | %(message)s',
    datefmt='%H:%M:%S'
)
log = logging.getLogger('01i_worldpop')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.dpi': 120, 'axes.titleweight': 'bold', 'font.size': 11})

OUTPUT_DIR = '/kaggle/working'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Column config
LAT_COL     = 'Latitude'
LON_COL     = 'Longitude'
STATION_COL = 'station_id'

# Helper function to find input file path in Kaggle
def find_file(filename, default_dir='/kaggle/working'):
    target = os.path.join(default_dir, filename)
    if os.path.exists(target):
        return target
    input_dir = '/kaggle/input'
    if os.path.exists(input_dir):
        for root, _, files in os.walk(input_dir):
            if filename in files:
                return os.path.join(root, filename)
    raise FileNotFoundError(f"File {filename} not found in input/working directories.")

In [ ]:
# Load base data to get unique stations
base_train_path = find_file('train_base.parquet')
base_val_path   = find_file('val_base.parquet')

train_base = pd.read_parquet(base_train_path)
val_base   = pd.read_parquet(base_val_path)
all_data   = pd.concat([train_base, val_base], ignore_index=True)

unique_stations = all_data.groupby(STATION_COL)[[LAT_COL, LON_COL]].first().reset_index()
log.info(f'Loaded unique stations: {len(unique_stations)}')

---
## Download WorldPop Raster

In [ ]:
url = 'https://data.worldpop.org/GIS/Population/Global_2015_2030/R2025A/2025/ZAF/v1/100m/constrained/zaf_pop_2025_CN_100m_R2025A_v1.tif'
local_path = '/tmp/zaf_pop_2025_CN_100m_R2025A_v1.tif'

if os.path.exists(local_path):
    log.info(f'File {local_path} already exists. Skipping download.')
else:
    log.info(f'Downloading WorldPop raster to {local_path}...')
    response = requests.get(url, stream=True, timeout=900)
    response.raise_for_status()
    
    total_size = int(response.headers.get('content-length', 0))
    block_size = 1024 * 1024  # 1MB chunk
    
    progress_bar = tqdm(total=total_size, unit='iB', unit_scale=True, desc='WorldPop')
    with open(local_path, 'wb') as f:
        for chunk in response.iter_content(chunk_size=block_size):
            if chunk:
                f.write(chunk)
                progress_bar.update(len(chunk))
    progress_bar.close()
    log.info('WorldPop raster downloaded successfully!')

---
## Extract Population Density

In [ ]:
def fetch_raster_mean_value(lat, lon, raster_src, buffer_m=1000):
    """Extract mean raster value within buffer around coordinates."""
    point = Point(lon, lat)
    deg_buffer = buffer_m / 111320.0  # approximate meters to degrees conversion
    buffered_geom = point.buffer(deg_buffer)
    
    try:
        # Crop raster to buffered area
        masked_img, _ = mask(raster_src, [buffered_geom], crop=True, nodata=0)
        valid_pixels = masked_img[0][masked_img[0] > 0]
        return float(valid_pixels.mean()) if valid_pixels.size > 0 else 0.0
    except Exception:
        return 0.0

In [ ]:
worldpop_df = unique_stations.copy()
total = len(worldpop_df)
log.info('Extracting WorldPop population density in 1km buffer...')

with rasterio.open(local_path) as src:
    worldpop_df['worldpop_mean_1km'] = worldpop_df.apply(
        lambda r: fetch_raster_mean_value(r[LAT_COL], r[LON_COL], src, 1000), axis=1
    )

display(worldpop_df.describe())

---
## Clean up and Save

In [ ]:
# Clean up large temp file to release Kaggle disk space
if os.path.exists(local_path):
    os.remove(local_path)
    log.info(f'Removed local TIFF file {local_path} to free disk space.')

out_path = f'{OUTPUT_DIR}/worldpop.parquet'
worldpop_df.to_parquet(out_path, index=False)
size_kb = os.path.getsize(out_path) / 1024
log.info(f'Saved: {out_path} ({size_kb:.1f} KB, {len(worldpop_df)} rows)')
print('\n=== DONE ===')
print('Output: worldpop.parquet')
print('Next: add this notebook output as dataset input for 01e')